# LinkedIn ML Selector (Classification)

This notebook is a classification-based version of the selector.

Workflow:
1. Load posts/followers/connections and build graph features.
2. Define labels:
   - `1` = scrappable (predefined seed influencers)
   - `0` = non-scrappable (manual blocklist)
3. Train and compare `XGBoost Classifier` vs `One-Class SVM`.
4. Evaluate with constant metrics and choose the better model.
5. Rank Top-K users for the next scraping batch.


In [25]:
import json
from pathlib import Path
from ast import literal_eval
import math

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.svm import OneClassSVM
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

ROOT_DIR = Path("/home/martin/technical-news")
NOTEBOOKS_DIR = ROOT_DIR / "notebooks"

CSV_PATH = NOTEBOOKS_DIR / "linkedin_scraping_data_20260203_191729.csv"
FOLLOWERS_PATH = NOTEBOOKS_DIR / "all_scraped_followers.json"
CONNECTIONS_PATH = NOTEBOOKS_DIR / "all_scraped_connections.json"


In [26]:
# Load raw data
posts_df = pd.read_csv(CSV_PATH)

with open(FOLLOWERS_PATH, "r", encoding="utf-8") as f:
    followers_data = json.load(f)
with open(CONNECTIONS_PATH, "r", encoding="utf-8") as f:
    connections_data = json.load(f)

print("Posts:", posts_df.shape)
print("Followers influencers:", followers_data.get("total_influencers"))
print("Connections influencers:", connections_data.get("total_influencers"))
posts_df.head(3)


Posts: (130, 21)
Followers influencers: 46
Connections influencers: 46


,post_url,activity_id,text,topic,supported_industry,keywords,reactions,comments,reposts,tagged_profiles,...,original_author,media_url,time_ago,scraped_at,engagement_score,author,influencer_name,influencer_title,followers_count,is_tech_related
0,https://www.linkedin.com/feed/update/urn:li:ac...,7422753343413895168,I built a tool to help me stay-up-date with ne...,['Generative AI'],['Technology'],ai,3370,130,187,[],...,NaN,https://media.licdn.com/dms/image/v2/D5622AQF5...,4d,2026-02-03T12:17:42.588520,3687,chiphuyen,Chip Huyen,AI x stuff,"303,333",True
1,https://www.linkedin.com/feed/update/urn:li:ac...,7358971409227792384,What people think will improve AI applications...,['Orchestration'],['Technology'],database,2132,107,184,[],...,NaN,https://media.licdn.com/dms/image/v2/D5622AQFs...,5m,2026-02-03T12:17:42.600294,2423,chiphuyen,Chip Huyen,AI x stuff,"303,333",True
2,https://www.linkedin.com/feed/update/urn:li:ac...,7389740123669721088,Had a great time chatting with Lenny Rachitsky...,['Generative AI'],['Technology'],api,766,57,41,['https://www.linkedin.com/in/lennyrachitsky/'],...,NaN,\N,3m,2026-02-03T12:17:42.595835,864,chiphuyen,Chip Huyen,AI x stuff,"303,333",True


In [27]:
# Helper functions

def linkedin_slug_from_url(url: str | None) -> str | None:
    if not isinstance(url, str) or not url:
        return None
    if "linkedin.com" not in url:
        return None
    try:
        parts = url.split("/")
        for i, p in enumerate(parts):
            if p == "in" and i + 1 < len(parts):
                return parts[i + 1] or None
        for p in reversed(parts):
            if p:
                return p
    except Exception:
        return None
    return None


def safe_parse_list(value):
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        v = value.strip()
        if v in ("", "[]", "\\N", "null", "None"):
            return []
        try:
            parsed = literal_eval(v)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return []
    return []


def normalize_slug(value: str | None) -> str | None:
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None
    if "linkedin.com" in s:
        slug = linkedin_slug_from_url(s)
        if slug:
            return str(slug).strip().lower().strip("/")
    return s.strip("/")



In [28]:
# Build graph from followers, connections, tags, reposts
G = nx.DiGraph()
seed_slugs = set()

for inf in followers_data.get("influencers", []):
    slug = linkedin_slug_from_url(inf.get("influencer_url"))
    if not slug:
        continue
    seed_slugs.add(slug)
    G.add_node(slug, name=inf.get("influencer"), url=inf.get("influencer_url"), role="influencer")

for inf in connections_data.get("influencers", []):
    slug = linkedin_slug_from_url(inf.get("influencer_url"))
    if not slug:
        continue
    seed_slugs.add(slug)
    if slug not in G:
        G.add_node(slug, name=inf.get("influencer"), url=inf.get("influencer_url"), role="influencer")

for inf in followers_data.get("influencers", []):
    src_slug = linkedin_slug_from_url(inf.get("influencer_url"))
    if not src_slug:
        continue
    for fol in inf.get("followers", []):
        dst_slug = linkedin_slug_from_url(fol.get("follower_url"))
        if not dst_slug:
            continue
        if dst_slug not in G:
            G.add_node(dst_slug, name=fol.get("follower_name"), url=fol.get("follower_url"), role="user", title=fol.get("follower_title"))
        G.add_edge(src_slug, dst_slug, type="FOLLOWER")

for inf in connections_data.get("influencers", []):
    src_slug = linkedin_slug_from_url(inf.get("influencer_url"))
    if not src_slug:
        continue
    for conn in inf.get("connections", []):
        dst_slug = linkedin_slug_from_url(conn.get("connection_url"))
        if not dst_slug:
            continue
        if dst_slug not in G:
            G.add_node(dst_slug, name=conn.get("connection_name"), url=conn.get("connection_url"), role="user", title=conn.get("connection_title"))
        G.add_edge(src_slug, dst_slug, type="CONNECTION")

if "tagged_profiles" in posts_df.columns:
    posts_df["tagged_profiles_parsed"] = posts_df["tagged_profiles"].apply(safe_parse_list)
else:
    posts_df["tagged_profiles_parsed"] = [[] for _ in range(len(posts_df))]

for _, row in posts_df.iterrows():
    author_slug = normalize_slug(row.get("author"))
    if author_slug and author_slug not in G:
        G.add_node(author_slug, name=row.get("influencer_name", author_slug), url=None, role="influencer" if author_slug in seed_slugs else "user")

    for tagged in row.get("tagged_profiles_parsed", []):
        dst_slug = normalize_slug(tagged)
        if not dst_slug:
            continue
        if dst_slug not in G:
            G.add_node(dst_slug, name=dst_slug, url=tagged, role="user")
        if author_slug:
            G.add_edge(author_slug, dst_slug, type="TAG")

    reposted = normalize_slug(row.get("reposted_profile"))
    if reposted:
        if reposted not in G:
            G.add_node(reposted, name=reposted, url=row.get("reposted_profile"), role="user")
        if author_slug:
            G.add_edge(author_slug, reposted, type="REPOST")

print(f"Total seed influencers: {len(seed_slugs)}")
print(f"Graph nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")
print("Edge type counts:", pd.Series([d.get("type") for _,_,d in G.edges(data=True)]).value_counts().to_dict())


Total seed influencers: 46
Graph nodes: 4282, edges: 4725
Edge type counts: {'CONNECTION': 2405, 'FOLLOWER': 2164, 'TAG': 156}


In [29]:
# Graph features
EDGE_TYPES = ["REPOST", "TAG", "CONNECTION", "FOLLOWER"]
G_undirected = G.to_undirected()
closeness_cent = nx.closeness_centrality(G_undirected)
pagerank = nx.pagerank(G, alpha=0.85)


def extract_relationship_features(slug: str) -> dict:
    counts = {f"num_{t.lower()}_edges": 0 for t in EDGE_TYPES}
    if slug not in G:
        return counts
    for src, _, data in G.in_edges(slug, data=True):
        if src not in seed_slugs:
            continue
        etype = data.get("type")
        key = f"num_{etype.lower()}_edges" if etype in EDGE_TYPES else None
        if key in counts:
            counts[key] += 1
    return counts

rows = []
for slug in G.nodes:
    feats = extract_relationship_features(slug)
    rows.append({
        "slug": normalize_slug(slug),
        "is_seed": normalize_slug(slug) in {normalize_slug(s) for s in seed_slugs},
        "total_relationships": sum(feats.values()),
        **feats,
        "closeness_centrality": closeness_cent.get(slug, 0.0),
        "pagerank": pagerank.get(slug, 0.0),
    })

features_df = pd.DataFrame(rows).drop_duplicates(subset=["slug"]).reset_index(drop=True)
print("Features shape:", features_df.shape)
features_df.head(5)


Features shape: (4282, 9)


,slug,is_seed,total_relationships,num_repost_edges,num_tag_edges,num_connection_edges,num_follower_edges,closeness_centrality,pagerank
0,chiphuyen,True,1,0,0,1,0,0.26,0.00
1,yann-lecun,True,0,0,0,0,0,0.24,0.00
2,alexxubyte,True,0,0,0,0,0,0.22,0.00
3,andrewyng,True,0,0,0,0,0,0.25,0.00
4,vanhoangkha,True,8,0,0,0,8,0.26,0.00


In [30]:
# Define classification labels: scrappable (1) vs non-scrappable (0)
NON_SCRAPPABLE_INPUTS = [
    "https://www.linkedin.com/in/-duncan-nguyen",
    "https://www.linkedin.com/in/amjad-almuwallad-929a7a130",
    "https://www.linkedin.com/in/bhatanusha",
    "https://www.linkedin.com/in/tuong-huynh-aba1a8275",
    "https://www.linkedin.com/in/er-romani-pathak-1849072a1",
    "https://www.linkedin.com/in/jon-gwynne-48b1878/",
    "https://www.linkedin.com/in/may-ann-pudoc-420655100",
    "https://www.linkedin.com/in/maryamreyhani",
    "https://www.linkedin.com/in/andrew-tattersall-72b6624a/",
    "https://www.linkedin.com/in/daisyhoa",
    "https://www.linkedin.com/in/linda-vi-b990a2223",
    "https://www.linkedin.com/in/ajay-arun-7371331a6",
    "https://www.linkedin.com/in/remi-el-ouazzane-6a1111",
    "https://www.linkedin.com/in/josemariasiota",
    "https://www.linkedin.com/in/jmartling",
    "https://www.linkedin.com/in/kevin-walling-56634773",
    "https://www.linkedin.com/in/annesorensen",
    "https://www.linkedin.com/in/bryan-stapleton",
    "https://www.linkedin.com/in/tueminh",
    "https://www.linkedin.com/in/martinbarnespresentations/",
    "https://www.linkedin.com/in/thomas-behncke",
    "https://www.linkedin.com/in/dawit-ghebrehiwet",
    "https://www.linkedin.com/in/cathrinhirling",
    "https://www.linkedin.com/in/dr-maike-neuhaus",
    "https://www.linkedin.com/in/dr-ines-anhorn-28778b1aa",
    "https://www.linkedin.com/in/duke-duong",
    "https://www.linkedin.com/in/thomascutts97/",
    "https://www.linkedin.com/in/taha-merchant-b40a57187/",
    "https://www.linkedin.com/in/brian-sii-044614123",
    "https://www.linkedin.com/in/ashley-quek-b636bb7b"
]

non_scrappable_slugs = {normalize_slug(x) for x in NON_SCRAPPABLE_INPUTS if normalize_slug(x)}
seed_slug_norm = {normalize_slug(x) for x in seed_slugs if normalize_slug(x)}

features_df = features_df.copy()
features_df["scrape_label"] = pd.NA
features_df.loc[features_df["slug"].isin(seed_slug_norm), "scrape_label"] = 1
features_df.loc[features_df["slug"].isin(non_scrappable_slugs), "scrape_label"] = 0

# Safety: if both sets overlap, non-scrappable wins
overlap = seed_slug_norm & non_scrappable_slugs
if overlap:
    features_df.loc[features_df["slug"].isin(overlap), "scrape_label"] = 0
    print(f"[WARN] overlap detected and forced to class 0: {sorted(overlap)}")

labeled_df = features_df[features_df["scrape_label"].notna()].copy()
labeled_df["scrape_label"] = labeled_df["scrape_label"].astype(int)

print("Labeled samples:", len(labeled_df))
print("Class distribution:")
print(labeled_df["scrape_label"].value_counts())

if labeled_df["scrape_label"].nunique() < 2:
    raise ValueError("Need both classes (1 and 0) in labeled_df.")


[WARN] overlap detected and forced to class 0: ['andrew-tattersall-72b6624a', 'martinbarnespresentations']
Labeled samples: 74
Class distribution:
scrape_label
1    44
0    30
Name: count, dtype: int64


In [31]:
# Train/test split for fair comparison
feature_cols = [
    "num_repost_edges",
    "num_tag_edges",
    "num_connection_edges",
    "num_follower_edges",
    "closeness_centrality",
    "pagerank",
]

X = labeled_df[feature_cols].values
y = labeled_df["scrape_label"].values
slugs = labeled_df["slug"].values

# Ensure each class has enough samples for stratified split
class_counts = labeled_df["scrape_label"].value_counts()
if class_counts.min() < 2:
    raise ValueError(f"Not enough samples per class for split: {class_counts.to_dict()}")

X_train, X_test, y_train, y_test, slug_train, slug_test = train_test_split(
    X, y, slugs, test_size=0.3, random_state=42, stratify=y
)

train_df = pd.DataFrame(X_train, columns=feature_cols)
train_df["label"] = y_train
test_df = pd.DataFrame(X_test, columns=feature_cols)
test_df["label"] = y_test
test_df["slug"] = slug_test

print("Train size:", len(train_df), "| Test size:", len(test_df))
print("Train class distribution:", train_df["label"].value_counts().to_dict())
print("Test class distribution:", test_df["label"].value_counts().to_dict())


Train size: 51 | Test size: 23
Train class distribution: {1: 30, 0: 21}
Test class distribution: {1: 14, 0: 9}


In [32]:
# LazyPredict benchmark on the same split
# Requires: pip install lazypredict

lazy_available = False
lazy_best_name = None
lazy_best_model = None
lazy_test_score = None

try:
    from lazypredict.Supervised import LazyClassifier, CLASSIFIERS

    lazy_clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    lazy_models_df, lazy_predictions_df = lazy_clf.fit(X_train, X_test, y_train, y_test)

    print('LazyPredict model leaderboard (classification):')
    display(lazy_models_df.head(20))

    if 'Balanced Accuracy' in lazy_models_df.columns:
        lazy_best_name = lazy_models_df['Balanced Accuracy'].idxmax()
    elif 'Accuracy' in lazy_models_df.columns:
        lazy_best_name = lazy_models_df['Accuracy'].idxmax()
    else:
        lazy_best_name = lazy_models_df.index[0]

    clf_map = {name: cls for name, cls in CLASSIFIERS}
    if lazy_best_name in clf_map:
        lazy_best_model = clf_map[lazy_best_name]()
        lazy_best_model.fit(X_train, y_train)

        if hasattr(lazy_best_model, 'predict_proba'):
            lazy_test_score = lazy_best_model.predict_proba(X_test)[:, 1]
        elif hasattr(lazy_best_model, 'decision_function'):
            raw = lazy_best_model.decision_function(X_test)
            rmin, rmax = raw.min(), raw.max()
            lazy_test_score = (raw - rmin) / (rmax - rmin + 1e-9)
        else:
            lazy_test_score = lazy_best_model.predict(X_test).astype(float)

        lazy_available = True
        print(f'Best LazyPredict model selected: {lazy_best_name}')
    else:
        print(f'Could not map LazyPredict model class for: {lazy_best_name}')

except ImportError:
    print('lazypredict is not installed. Install with: pip install lazypredict')

except Exception as e:
    print(f'LazyPredict run failed: {e}')


  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 30, number of negative: 21
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000159 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 26
[LightGBM] [Info] Number of data points in the train set: 51, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,588235 -> initscore=0,356675
[LightGBM] [Info] Start training from score 0,356675
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Light

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
AdaBoostClassifier,1.00,1.00,1.00,1.00,0.04
KNeighborsClassifier,1.00,1.00,1.00,1.00,0.01
SVC,1.00,1.00,1.00,1.00,0.01
RandomForestClassifier,1.00,1.00,1.00,1.00,0.05
NuSVC,1.00,1.00,1.00,1.00,0.01
BaggingClassifier,1.00,1.00,1.00,1.00,0.01
LabelSpreading,1.00,1.00,1.00,1.00,0.00
LabelPropagation,1.00,1.00,1.00,1.00,0.00
LGBMClassifier,1.00,1.00,1.00,1.00,0.03


Best LazyPredict model selected: AdaBoostClassifier


In [33]:
# Train XGBoost classifier
pos = max(int((y_train == 1).sum()), 1)
neg = max(int((y_train == 0).sum()), 1)
scale_pos_weight = neg / pos

xgb_clf = XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight,
)
xgb_clf.fit(X_train, y_train)

xgb_test_score = xgb_clf.predict_proba(X_test)[:, 1]


In [34]:
# Train One-Class SVM using only positive train samples
X_train_pos = X_train[y_train == 1]
if len(X_train_pos) < 5:
    raise ValueError("Not enough positive samples to train OCSVM reliably.")

ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.1)
ocsvm.fit(X_train_pos)

ocsvm_test_raw = ocsvm.decision_function(X_test)
# normalize for easier cross-model comparison
oc_min, oc_max = ocsvm_test_raw.min(), ocsvm_test_raw.max()
ocsvm_test_score = (ocsvm_test_raw - oc_min) / (oc_max - oc_min + 1e-9)


In [35]:
# Evaluation metrics for model comparison

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    topk = y_true[order][:k]
    return float(np.mean(topk)) if len(topk) else 0.0


def recall_at_k(y_true, scores, k):
    total_pos = int(np.sum(y_true == 1))
    if total_pos == 0:
        return 0.0
    order = np.argsort(-scores)
    topk = y_true[order][:k]
    return float(np.sum(topk == 1) / total_pos)


def ndcg_at_k_binary(y_true, scores, k):
    order = np.argsort(-scores)
    rel = y_true[order][:k].astype(float)
    dcg = np.sum(rel / np.log2(np.arange(2, len(rel) + 2)))

    ideal = np.sort(y_true)[::-1][:k].astype(float)
    idcg = np.sum(ideal / np.log2(np.arange(2, len(ideal) + 2)))
    return float(dcg / idcg) if idcg > 0 else 0.0


def evaluate_model(y_true, scores, model_name, k_values):
    rows = []
    auc = roc_auc_score(y_true, scores)
    ap = average_precision_score(y_true, scores)
    for k in k_values:
        rows.append({
            "model": model_name,
            "k": int(k),
            "roc_auc": float(auc),
            "avg_precision": float(ap),
            "precision_at_k": precision_at_k(y_true, scores, k),
            "recall_at_k": recall_at_k(y_true, scores, k),
            "ndcg_at_k": ndcg_at_k_binary(y_true, scores, k),
            "yield_at_k": float(np.sum(y_true[np.argsort(-scores)][:k] == 1)),
        })
    return pd.DataFrame(rows)


max_k = len(y_test)
k_values = [k for k in [5, 10, 20] if k <= max_k]
if not k_values:
    k_values = [max_k]

metrics_xgb = evaluate_model(y_test, xgb_test_score, "XGBoost", k_values)
metrics_oc = evaluate_model(y_test, ocsvm_test_score, "OCSVM", k_values)
frames = [metrics_xgb, metrics_oc]

if 'lazy_test_score' in globals() and lazy_test_score is not None:
    metrics_lazy = evaluate_model(y_test, lazy_test_score, f"LazyBest:{lazy_best_name}", k_values)
    frames.append(metrics_lazy)

metrics_df = pd.concat(frames, ignore_index=True).sort_values(["k", "model"])

print("Model comparison metrics:")
display(metrics_df)


Model comparison metrics:


,model,k,roc_auc,avg_precision,precision_at_k,recall_at_k,ndcg_at_k,yield_at_k
6,LazyBest:AdaBoostClassifier,5,1.00,1.00,1.00,0.36,1.00,5.00
3,OCSVM,5,0.95,0.97,1.00,0.36,1.00,5.00
0,XGBoost,5,1.00,1.00,1.00,0.36,1.00,5.00
7,LazyBest:AdaBoostClassifier,10,1.00,1.00,1.00,0.71,1.00,10.00
4,OCSVM,10,0.95,0.97,1.00,0.71,1.00,10.00
1,XGBoost,10,1.00,1.00,1.00,0.71,1.00,10.00
8,LazyBest:AdaBoostClassifier,20,1.00,1.00,0.70,1.00,1.00,14.00
5,OCSVM,20,0.95,0.97,0.70,1.00,0.99,14.00
2,XGBoost,20,1.00,1.00,0.70,1.00,1.00,14.00


In [36]:
# Decide best model from metrics
summary = (
    metrics_df.groupby("model", as_index=False)
    .agg(
        mean_roc_auc=("roc_auc", "mean"),
        mean_avg_precision=("avg_precision", "mean"),
        mean_precision_at_k=("precision_at_k", "mean"),
        mean_recall_at_k=("recall_at_k", "mean"),
        mean_ndcg_at_k=("ndcg_at_k", "mean"),
        mean_yield_at_k=("yield_at_k", "mean"),
    )
)

# Winner rule for scraping ranking: ndcg first, then recall, then roc_auc
summary = summary.sort_values(
    ["mean_ndcg_at_k", "mean_recall_at_k", "mean_roc_auc"],
    ascending=[False, False, False],
).reset_index(drop=True)

best_model = summary.loc[0, "model"]
print("Best model for top-k scraping selection:", best_model)
print("Summary metrics:")
display(summary)


Best model for top-k scraping selection: LazyBest:AdaBoostClassifier
Summary metrics:


,model,mean_roc_auc,mean_avg_precision,mean_precision_at_k,mean_recall_at_k,mean_ndcg_at_k,mean_yield_at_k
0,LazyBest:AdaBoostClassifier,1.00,1.00,0.90,0.69,1.00,9.67
1,XGBoost,1.00,1.00,0.90,0.69,1.00,9.67
2,OCSVM,0.95,0.97,0.90,0.69,1.00,9.67


In [37]:
# Score all users and select top-k for next batch
all_X = features_df[feature_cols].values
features_df = features_df.copy()
features_df["xgb_scrape_score"] = xgb_clf.predict_proba(all_X)[:, 1]

oc_all_raw = ocsvm.decision_function(all_X)
oc_a, oc_b = oc_all_raw.min(), oc_all_raw.max()
features_df["ocsvm_scrape_score"] = (oc_all_raw - oc_a) / (oc_b - oc_a + 1e-9)

lazy_score_col = None
if 'lazy_available' in globals() and lazy_available and (lazy_best_model is not None):
    if hasattr(lazy_best_model, 'predict_proba'):
        lazy_all_score = lazy_best_model.predict_proba(all_X)[:, 1]
    elif hasattr(lazy_best_model, 'decision_function'):
        raw = lazy_best_model.decision_function(all_X)
        rmin, rmax = raw.min(), raw.max()
        lazy_all_score = (raw - rmin) / (rmax - rmin + 1e-9)
    else:
        lazy_all_score = lazy_best_model.predict(all_X).astype(float)

    lazy_score_col = "lazy_scrape_score"
    features_df[lazy_score_col] = lazy_all_score

if best_model == "XGBoost":
    score_col = "xgb_scrape_score"
elif best_model == "OCSVM":
    score_col = "ocsvm_scrape_score"
elif best_model.startswith("LazyBest") and lazy_score_col is not None:
    score_col = lazy_score_col
else:
    score_col = "xgb_scrape_score"

candidates_df = features_df[
    (~features_df["is_seed"]) &
    (features_df["total_relationships"] > 0) &
    (~features_df["slug"].isin(non_scrappable_slugs))
].copy()

candidates_df = candidates_df.sort_values(score_col, ascending=False)

names, urls, roles = [], [], []
for slug in candidates_df["slug"]:
    node = G.nodes.get(slug, {})
    names.append(node.get("name"))
    urls.append(node.get("url"))
    roles.append(node.get("role", "user"))

candidates_df["name"] = names
candidates_df["url"] = urls
candidates_df["role"] = roles

TOP_K = 60
cols = [
    "slug", "name", "url", "role", score_col,
    "num_repost_edges", "num_tag_edges", "num_connection_edges", "num_follower_edges",
]

print(f"Top {TOP_K} users for next scraping batch (selected by {best_model}):")
display(candidates_df[cols].head(TOP_K))


Top 60 users for next scraping batch (selected by LazyBest:AdaBoostClassifier):


,slug,name,url,role,lazy_scrape_score,num_repost_edges,num_tag_edges,num_connection_edges,num_follower_edges
371,tuannguyen5,Tuan Nguyen,https://www.linkedin.com/in/tuannguyen5,user,0.62,0,0,0,10
54,quang-le-kim-846b3823b,Quang Le Kim,https://www.linkedin.com/in/quang-le-kim-846b3...,user,0.62,0,0,0,9
145,quan-dang,Quan Dang,https://www.linkedin.com/in/quan-dang,user,0.62,0,0,0,8
4277,vicyaa,vicyaa,https://www.linkedin.com/in/vicyaa/,user,0.59,0,1,0,0
4276,maryam-yasaei,maryam-yasaei,https://www.linkedin.com/in/maryam-yasaei/,user,0.59,0,1,0,0
3994,zachjensz,Zach Jensz 🚀,https://www.linkedin.com/in/zachjensz,user,0.59,0,1,0,0
4231,samuelphilipos,samuelphilipos,https://www.linkedin.com/in/samuelphilipos/,user,0.59,0,1,0,0
4197,ckpearce,ckpearce,https://www.linkedin.com/in/ckpearce/,user,0.59,0,1,0,0
4250,carinaparisella,carinaparisella,https://www.linkedin.com/in/carinaparisella/,user,0.59,0,1,0,0
4251,jim-hogan-innovation,jim-hogan-innovation,https://www.linkedin.com/in/jim-hogan-innovation/,user,0.59,0,1,0,0
